# 01 — Python Engineering Refresher

**Goal:** Review Python features essential for production NLP code.

We'll cover:
1. Type hints and dataclasses
2. List/dict comprehensions and generators
3. File I/O and context managers
4. Error handling best practices
5. Functions as first-class objects
6. A mini resume parser using what we learn

This chapter is a *language* refresher, not an NLP one: the Python idioms production NLP code is written in. Typed functions, dataclasses, comprehensions, generators, context managers, narrow exception handling, and function composition all reappear throughout the series — usually without explanation.

**Why it matters for resumes / ATS:** the mini resume parser at the end of this chapter is the project's first artifact, built exclusively from these idioms. Interviewers probe exactly this material (type hints, the mutable-default trap, generator memory, pipeline composition) when they ask you to walk through parsing code — and this chapter is where you build that answer.

## 1. Type Hints — Self-Documenting Code

Type hints document *intent* at the signature: readers and tools (`mypy`, IDE autocomplete) see what a function consumes and returns without reading its body. They are not enforced at runtime — they are contracts for humans and linters.

**What the code does:**
- `extract_skills` (untyped) vs `extract_skills_typed(text: str) -> List[str]`: same job, but the typed version also strips empties and declares its contract up front.
- `Optional[str]` is the honest signature for a function that may return `None` — the stored output shows both branches: a match returns the email, no match returns `None`.
- The email regex (`[\w.+-]+@[\w-]+\.[\w.]+`) is Ch. 03 material; here it just feeds the typing lesson.

**Try it:** the two `Email found:` lines in the output — one match, one `None` — show exactly why the return type must be `Optional`.

In [1]:
from typing import List, Dict, Optional, Tuple, Union

# Without type hints — what does this return?
def extract_skills(text):
    return text.split(",")

# With type hints — clear intent
def extract_skills_typed(text: str) -> List[str]:
    """Extract comma-separated skills from text."""
    return [s.strip() for s in text.split(",") if s.strip()]

# Optional and Union for nullable fields
def find_email(text: str) -> Optional[str]:
    """Return email if found, None otherwise."""
    import re
    match = re.search(r"[\w.+-]+@[\w-]+\.[\w.]+", text)
    return match.group(0) if match else None

print(extract_skills_typed("Python, NLP, Machine Learning"))
print(f"Email found: {find_email('Contact: john@example.com')}")
print(f"Email found: {find_email('No email here')}")

['Python', 'NLP', 'Machine Learning']
Email found: john@example.com
Email found: None


## 2. Dataclasses — Clean Data Containers

A `@dataclass` auto-generates `__init__`, `__repr__`, and `__eq__` from field annotations — a plain class would need ~15 lines of boilerplate for the same result. Mutable defaults must go through `field(default_factory=list)`, or every instance shares one list (the classic Python gotcha).

**What the code does:**
- `Skill` (name, category, confidence) and `Resume` (name, email, skills) model a parsed resume as typed records.
- `Resume.add_skill(...)` appends a `Skill`; `top_skills(threshold=0.8)` filters on confidence.
- The stored output shows the auto-generated `repr` and `Top skills: ['Python', 'NLP']` — the 0.95/0.88-confidence skills clear the 0.8 bar while Java (0.6) drops out.

**Why it matters:** structured records like these are what every later extractor emits — dataclasses are this project's standard container.

In [ ]:
from dataclasses import dataclass, field

@dataclass
class Skill:
    name: str
    category: str = "technical"
    confidence: float = 0.0

@dataclass
class Resume:
    name: str
    email: Optional[str] = None
    skills: List[Skill] = field(default_factory=list)

    def add_skill(self, name: str, category: str = "technical",
                confidence: float = 0.0) -> None:
        self.skills.append(Skill(name, category, confidence))

    def top_skills(self, threshold: float = 0.8) -> List[Skill]:
        return [s for s in self.skills if s.confidence >= threshold]

# Use it
resume = Resume("Srivatsa", "srivatsa@example.com")
resume.add_skill("Python", confidence=0.95)
resume.add_skill("NLP", confidence=0.88)
resume.add_skill("Java", confidence=0.60)
print(resume)
print(f"Top skills: {[s.name for s in resume.top_skills()]}")

Resume(name='Srivatsa', email='srivatsa@example.com', skills=[Skill(name='Python', category='technical', confidence=0.95), Skill(name='NLP', category='technical', confidence=0.88), Skill(name='Java', category='technical', confidence=0.6)])
Top skills: ['Python', 'NLP']


## 3. List Comprehensions — Fast & Readable

Comprehensions are loops that *return a value* — and in CPython they run in C-accelerated bytecode, faster than an equivalent `for` + `append` and usually clearer. Dict and set comprehensions extend the same syntax to the other container types.

**What the code does:**
- The loop and the comprehension produce identical output (`['Python', 'NLP', 'Machine Learning']`) after stripping whitespace and dropping the empty string.
- `skill_lengths = {s: len(s) for s in cleaned}` builds a length lookup — the stored output shows `'Machine Learning': 16`, spaces and all.
- The set comprehension collapses every character across all skills: the output's **15 unique chars** is the deduplicated alphabet of the skill list.

**Try it:** compare line counts of the two versions — then note the comprehension is also the one you can nest inside a larger expression.

In [3]:
# Traditional loop
skills = [" Python ", "NLP ", " Machine Learning ", ""]
cleaned = []
for s in skills:
    s = s.strip()
    if s:
        cleaned.append(s)
print("Loop:", cleaned)

# List comprehension (faster, cleaner)
cleaned = [s.strip() for s in skills if s.strip()]
print("Comprehension:", cleaned)

# Dict comprehension
skill_lengths = {s: len(s) for s in cleaned}
print("Lengths:", skill_lengths)

# Set comprehension
unique_chars = {c for s in cleaned for c in s.lower()}
print(f"Unique chars across skills: {len(unique_chars)}")

Loop: ['Python', 'NLP', 'Machine Learning']
Comprehension: ['Python', 'NLP', 'Machine Learning']
Lengths: {'Python': 6, 'NLP': 3, 'Machine Learning': 16}
Unique chars across skills: 15


## 4. Generators — Memory-Efficient Processing

Generators produce values *lazily*: nothing is computed until `next()` is called, so a generator over a huge corpus costs constant memory. `yield` turns any function into a generator; generator expressions (`(x for x in ...)`) do the same inline.

**What the code does:**
- `read_chunks` yields 100-character slices of a 30 KB string; `type(chunks)` reports `<class 'generator'>` — no list was ever materialized.
- `next(chunks)` pulls one chunk at a time, as the two printed chunks show.
- `sum(1 for chunk in ... if "NLP" in chunk)` counts matching chunks without building any intermediate list — the whole point.

**Why it matters:** resumes, job-description corpora, and training text are processed chunk-by-chunk exactly this way when they don't fit in memory.

In [4]:
def read_chunks(text: str, chunk_size: int = 50):
    """Yield chunks of text without loading everything into memory."""
    for i in range(0, len(text), chunk_size):
        yield text[i:i + chunk_size]

# Generator expression (lazy)
large_text = "Python is great for NLP. " * 1000
chunks = read_chunks(large_text, 100)
print(f"Type: {type(chunks)}")
print(f"First chunk: {next(chunks)}")
print(f"Second chunk: {next(chunks)}")

# Count without building a list
word_count = sum(1 for chunk in read_chunks(large_text, 50) if "NLP" in chunk)
print(f"Chunks containing 'NLP': {word_count}")

Type: <class 'generator'>
First chunk: Python is great for NLP. Python is great for NLP. Python is great for NLP. Python is great for NLP. 
Second chunk: Python is great for NLP. Python is great for NLP. Python is great for NLP. Python is great for NLP. 
Chunks containing 'NLP': 500


## 5. File I/O — Reading Resumes

The `with` statement guarantees the file is closed even when the body raises — manual `f.close()` in `try/finally` is the error-prone alternative. Files also stream: `for line in f` reads lazily, one line at a time, which is how you process files larger than RAM.

**What the code does:**
- Writes a sample resume with `open(..., 'w')`, then reads it back.
- Deliberately demonstrates the **file-pointer trap**: after `f.read()` consumes the stream, `f.readlines()` returns `[]` — the inline comment flags it as a demo of stateful file objects.
- The correct read reopens the file and iterates lines lazily; the stored output prints each stripped line of the resume.

**Try it:** delete the second `with` block and rerun — you will see the empty `readlines()` result the notebook warns about.

In [7]:
# Context manager handles cleanup automatically
sample_text = """Name: John Doe
Email: john@example.com
Skills: Python, NLP, Machine Learning, TensorFlow
Experience: 5 years as Data Scientist
"""

# Write sample
with open("sample_resume.txt", "w") as f:
    f.write(sample_text)

# Read it back
with open("sample_resume.txt", "r") as f:
    content = f.read()
    lines = f.readlines()  # already consumed! (demonstrating file pointer)

# Read properly
with open("sample_resume.txt", "r") as f:
    for line in f:  # lazy iteration
        print(repr(line.strip()))

'Name: John Doe'
'Email: john@example.com'
'Skills: Python, NLP, Machine Learning, TensorFlow'
'Experience: 5 years as Data Scientist'


## 6. Error Handling — Robust Extraction

Production parsing assumes hostile input: `None`, wrong types, empty strings, malformed lines. The pattern is *raise early with a specific exception, catch narrowly, degrade gracefully* — and never swallow `Exception` silently.

**What the code does:**
- `safe_extract_email` validates first: `TypeError` for non-strings, `ValueError` for empty text, then the regex.
- The `except (TypeError, ValueError)` branch logs a warning and returns `None`; a catch-all `except Exception` is the last resort.
- The stored output walks the three cases: a real email extracts; `""` prints `Warning: Empty text` then `None`; `123` prints `Warning: Expected string, got <class 'int'>` then `None`.

**Why it matters:** every extractor in Ch. 03–13 inherits this shape — bad input must degrade to a missing field, never crash the pipeline.

In [8]:
def safe_extract_email(text: str) -> Optional[str]:
    """Extract email with proper error handling."""
    try:
        if not isinstance(text, str):
            raise TypeError(f"Expected string, got {type(text)}")
        if not text.strip():
            raise ValueError("Empty text")
        import re
        match = re.search(r"[\w.+-]+@[\w-]+\.[\w.]+", text)
        return match.group(0) if match else None
    except (TypeError, ValueError) as e:
        print(f"Warning: {e}")
        return None
    except Exception as e:
        print(f"Unexpected error: {e}")
        return None

# Test cases
print(safe_extract_email("Email: test@example.com"))    # ✓
print(safe_extract_email(""))                            # Warning
print(safe_extract_email(123))                           # Warning

test@example.com
None
None


## 7. Functions as Objects — Pipeline Pattern

Functions are values: passable as arguments, storable, returnable. That makes *composition* trivial — `build_pipeline` chains any number of callables into one function, the same idea behind `sklearn.pipeline` and spaCy's component pipeline.

**What the code does:**
- `clean_text` (strip + lowercase) and `remove_numbers` (regex `\d+` removal) are plain functions.
- `build_pipeline(*functions)` returns a closure applying each function in order; `cleaner = build_pipeline(clean_text, remove_numbers)` fixes the pipeline once and reuses it.
- The stored output `Cleaned: 'hello  world!'` shows the double-space artifact left after digits are removed — a real reminder that composed transforms interact.

**Try it:** add a third function (e.g. collapse repeated spaces) to the pipeline and watch the artifact disappear.

In [9]:
from typing import Callable, Any

# Functions are first-class — pass them around
def clean_text(text: str) -> str:
    return text.strip().lower()

def remove_numbers(text: str) -> str:
    import re
    return re.sub(r"\d+", "", text)

def build_pipeline(*functions: Callable) -> Callable:
    """Compose multiple processing functions."""
    def pipeline(text: str) -> str:
        result = text
        for func in functions:
            result = func(result)
        return result
    return pipeline

# Build and use
cleaner = build_pipeline(clean_text, remove_numbers)
result = cleaner("  Hello 123 World!  ")
print(f"Cleaned: '{result}'")

Cleaned: 'hello  world!'


## 8. Mini Resume Info Extractor

Putting it all together:

This cell is the chapter's payoff: a first working resume parser built only from the idioms above. It is deliberately *heuristic* — name = first line with 2–3 words, contact = regexes, skills = substring match against a known list — and that honesty matters: it works on clean samples and stumbles on messy reality, which is exactly why the rest of Part I exists.

**What the code does:**
- `ContactInfo` / `SimpleResume` dataclasses hold the structured result.
- `extract_name` takes the first non-empty line if it has 2–3 words; `extract_contact` regexes email and phone (`[+]?[\d\s()-]{7,}`); `extract_skills` case-folds the text and checks membership against `known_skills`.
- The stored output is the full parse: **Srivatsa Gorti**, `srivatsa@email.com`, `+91-9876543210`, and 5 of 10 known skills — `Java`, `Spark`, `Docker`, `Kubernetes` correctly absent.

**Try it:** feed it a resume whose first line is a title ("Senior Data Scientist") — the heuristic returns the title as the name. That failure mode motivates everything from Ch. 03 onward.

In [10]:
import re
from dataclasses import dataclass, field
from typing import List, Optional

@dataclass
class ContactInfo:
    name: Optional[str] = None
    email: Optional[str] = None
    phone: Optional[str] = None

@dataclass
class SimpleResume:
    raw_text: str
    contact: ContactInfo = field(default_factory=ContactInfo)
    skills: List[str] = field(default_factory=list)

def extract_name(text: str) -> Optional[str]:
    """Simple heuristic: first line often has the name."""
    first_line = text.strip().split("\n")[0].strip()
    if first_line and len(first_line.split()) in [2, 3]:
        return first_line
    return None

def extract_contact(text: str) -> ContactInfo:
    info = ContactInfo()
    info.name = extract_name(text)
    email_match = re.search(r"[\w.+-]+@[\w-]+\.[\w.]+", text)
    info.email = email_match.group(0) if email_match else None
    phone_match = re.search(r"[+]?[\d\s()-]{7,}", text)
    info.phone = phone_match.group(0).strip() if phone_match else None
    return info

def extract_skills(text: str, skill_list: List[str]) -> List[str]:
    """Find known skills in text."""
    text_lower = text.lower()
    found = []
    for skill in skill_list:
        if skill.lower() in text_lower:
            found.append(skill)
    return found

# Test it
sample = """Srivatsa Gorti
srivatsa@email.com | +91-9876543210

Professional Summary
Data scientist with 3 years experience in Python, NLP and ML.

Skills: Python, NLP, Machine Learning, SQL, TensorFlow
"""

known_skills = ["Python", "NLP", "Machine Learning", "SQL", "TensorFlow",
                "Java", "Spark", "Docker", "Kubernetes"]

resume = SimpleResume(raw_text=sample)
resume.contact = extract_contact(sample)
resume.skills = extract_skills(sample, known_skills)

print(f"Name:   {resume.contact.name}")
print(f"Email:  {resume.contact.email}")
print(f"Phone:  {resume.contact.phone}")
print(f"Skills: {resume.skills}")

Name:   Srivatsa Gorti
Email:  srivatsa@email.com
Phone:  +91-9876543210
Skills: ['Python', 'NLP', 'Machine Learning', 'SQL', 'TensorFlow']


## Summary

Today you learned:
- ✅ Type hints and dataclasses for clean code
- ✅ List/dict comprehensions (faster than loops)
- ✅ Generators for memory efficiency
- ✅ Context managers for safe I/O
- ✅ Error handling patterns
- ✅ Functions as first-class objects → pipeline pattern

These patterns will be used in every notebook going forward.

These patterns are the vocabulary of every notebook in this series: dataclasses for records, comprehensions for transforms, generators for streaming, context managers for I/O, narrow exceptions for hostile input, and composition for pipelines.

The next chapter swaps the *language* for the *data* layer — Ch. 02, Pandas & NumPy, vectorizes all of this over candidate and job-description tables.

## Key Insight

**Idiomatic Python is the difference between a demo and a pipeline.**

Every pattern in this chapter exists because production parsing code is typed, lazy, defensive, and composable: dataclasses give extractors a structured contract, generators keep corpora streamable, narrow exceptions keep hostile input from crashing a batch run, and function composition turns a sequence of transforms into a reusable pipeline — the exact shape of the ATS stack in Part II. This is also the code style later notebooks assume without comment. Next, the data layer those pipelines operate on — Ch. 02, Pandas & NumPy.